# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIRˆ² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print metadata overview
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Number of authors: {len(metadata.author) if hasattr(metadata, 'author') else 0}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets in the dataset
print('Available record sets:')
record_sets = []
for rs in dataset.metadata.recordSet:
    print(f"- Name: {getattr(rs, 'name', '(no name)')}")
    print(f"  @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', 'N/A')}")
    record_sets.append(rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None))
    if hasattr(rs, 'field'):
        print('  Fields:')
        for field in rs.field:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', 'N/A')
            print(f"    - {getattr(field, 'name', '(no name)')}: {field_id}")
    print()

# If no recordSet in metadata, try to find via the API directly
if not record_sets:
    all_rs = list(dataset.record_sets())
    for rs in all_rs:
        print(f"- Name: {getattr(rs, 'name', '(no name)')}")
        print(f"  @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', 'N/A')}")
        record_sets.append(rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None))

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Record set and field `@id`s are used.

In [ ]:
# -- Automatic detection of record sets for extraction --
if not record_sets:
    # Fallback: discover via dataset.record_sets API
    record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
                  for rs in list(dataset.record_sets())]
    record_sets = [rs for rs in record_sets if rs is not None]

print('Record sets to extract:')
print(record_sets)

dataframes = {}

for record_set_id in record_sets:
    print(f"\nExtracting {record_set_id} ...")
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    if df.shape[0] == 0:
        print('  (No records found)')
    else:
        print(f"  {df.shape[0]} records.")
        print(f"  Columns: {df.columns.tolist()}")
    dataframes[record_set_id] = df

# Show sample data for first nonempty record set
main_set = None
for k, v in dataframes.items():
    if not v.empty:
        main_set = k
        break

if main_set:
    print(f"\nDisplaying first records from: {main_set}")
    display(dataframes[main_set].head())
else:
    print("No records to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
if not main_set:
    print("No data for EDA.")
else:
    df = dataframes[main_set]
    print(f"Columns: {df.columns.tolist()}")
    
    # Try to infer a numeric field ('age', 'interval', etc)
    import re
    numeric_candidates = [col for col in df.columns if re.search(r'age|interval|months|years|count|num', col, re.I)]
    if not numeric_candidates:
        numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric field candidates: {numeric_candidates}")
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        # Filter records where numeric_field > threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize the numeric field
        norm_col = numeric_field + '_normalized'
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
        # Try to group by a categorical field
        group_candidates = [col for col in df.columns if df[col].nunique() < min(len(df)//4, 10) and col != numeric_field]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped means by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_set:
    print('No data to visualize.')
else:
    df = dataframes[main_set]
    if 'numeric_field' in locals():
        # Distribution
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field], kde=True)
        plt.title(f"Distribution of '{numeric_field}'")
        plt.xlabel(numeric_field)
        plt.show()
        # Boxplot by group if available
        if 'group_field' in locals():
            plt.figure(figsize=(10, 5))
            sns.boxplot(x=df[group_field], y=df[numeric_field])
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=30)
            plt.show()

## 6. Conclusion
In this notebook, we have demonstrated metadata exploration, tabular data extraction, simple filtering, normalization, and visual analytics using the `mlcroissant` library on the FAIRˆ² clinical oncology dataset defined by a Croissant schema.

_For more advanced analytics or to reference fields precisely, always use the entity `@id` as in this notebook._